# Timings

In [1]:
from collections import Counter
from itertools import groupby
from math import isqrt
from numbers import Number
from operator import itemgetter
from typing import Iterator

In [2]:
def counter_method(z: int) -> tuple[tuple[int, int], ...]:
    ctr: Counter[int] = Counter()

    # if `z` is even, divide `z` by 2 and keep track how many divisions before `z` becomes odd
    while z & 1 == 0:
        ctr.update((2,))
        z //= 2

    # odd numbers from 3 to sqrt(z); N.b., will never get to the end of
    # this range because `z` is iteratively decremented
    candidates: Iterator[int] = (n for n in range(3, isqrt(z) + 1, 2))
    for o in candidates:
        # call z/o the name q (quotient). If q is an integer, then o is a divisor of z
        while (q := z / o).is_integer():
            # record o as a factor, or increment the count of o as a factor
            ctr.update((o,))
            # proceed with q as the new z
            z = int(q)

    if z > 1:
        ctr.update((z,))
    # canonical form sorted by prime, not by prime's exponent
    # return tuple(sorted(ctr.most_common(), key=itemgetter(1)))
    return tuple(ctr.items())

In [3]:
def list_method(z: int) -> tuple[tuple[int, int], ...]:
    ctr: list[int] = [None for _ in range(z)]
    i: int = 0

    while z & 1 == 0:
        ctr[i] = 2
        i += 1
        z //= 2
    
    candidates: Iterator[int] = (n for n in range(3, isqrt(z) + 1, 2))
    for o in candidates:
        while (q := z / o).is_integer():
            # record o as a factor, or increment the count of o as a factor
            ctr[i] = o
            i += 1
            # proceed with q as the new z
            z = int(q)

    if z > 1:
        ctr[i] = z
    
    return tuple(
        (el, ctr.count(el))
        for i, el in enumerate(ctr)
        if el != ctr[i-1] and el is not None
    )

In [4]:
def gby_method(z: int) -> tuple[tuple[int, int], ...]:
    """This one yields."""
    def generate_prime_divisors(z: int) -> Iterator[int]:
        while z & 1 == 0:
            yield 2
            z //= 2
        
        candidates: Iterator[int] = (n for n in range(3, isqrt(z) + 1, 2))
        for o in candidates:
            while (q := z / o).is_integer():
                # record o as a factor, or increment the count of o as a factor
                yield o
                # proceed with q as the new z
                z = int(q)

        if z > 1:
            yield z
    
    return tuple(
        (p, sum(1 for _ in group))
        for p, group in groupby(generate_prime_divisors(z))
    )

## Small Prime
Let's start with 5

In [10]:
Z = 5
expected = ((Z, 1),)

### `collections.Counter` Method

In [11]:
assert counter_method(Z) == expected
assert list_method(Z) == expected
assert gby_method(Z) == expected

In [12]:
%%timeit -r 30
counter_method(z=Z)

1.28 μs ± 6.96 ns per loop (mean ± std. dev. of 30 runs, 1,000,000 loops each)


### `list` Method

In [13]:
%%timeit -r 30
list_method(Z)

1.33 μs ± 7.19 ns per loop (mean ± std. dev. of 30 runs, 1,000,000 loops each)


### `itertools.groupby` Method

In [14]:
%%timeit -r 30
gby_method(Z)

1.16 μs ± 6.8 ns per loop (mean ± std. dev. of 30 runs, 1,000,000 loops each)


## Small Composite

In [15]:
Z = 14
expected = ((2, 1), (7, 1))

In [16]:
assert counter_method(Z) == expected
assert list_method(Z) == expected
assert gby_method(Z) == expected

### `collections.Counter` Method

In [17]:
%%timeit -r 30
counter_method(Z)

1.7 μs ± 9.28 ns per loop (mean ± std. dev. of 30 runs, 1,000,000 loops each)


### `list` Method

In [18]:
%%timeit -r 30
list_method(Z)

2.16 μs ± 22.6 ns per loop (mean ± std. dev. of 30 runs, 100,000 loops each)


### `itertools.groupby` Method

In [19]:
%%timeit -r 30
gby_method(Z)

1.51 μs ± 7.43 ns per loop (mean ± std. dev. of 30 runs, 1,000,000 loops each)


## Six-Digit Prime

In [20]:
Z = 666_667
expected = ((Z, 1),)

In [21]:
assert counter_method(Z) == expected
assert list_method(Z) == expected
assert gby_method(Z) == expected

In [22]:
%%timeit -r 30
counter_method(Z)

28.1 μs ± 377 ns per loop (mean ± std. dev. of 30 runs, 10,000 loops each)


In [23]:
%%timeit -r 30
list_method(Z)

54.1 ms ± 861 μs per loop (mean ± std. dev. of 30 runs, 10 loops each)


In [24]:
%%timeit -r 30
gby_method(Z)

27.9 μs ± 366 ns per loop (mean ± std. dev. of 30 runs, 10,000 loops each)


## Six-Digit Highly-Composite
See https://oeis.org/A002182

In [25]:
Z = 110_880
expected = ((2, 5), (3, 2), (5, 1), (7,1), (11, 1))

In [26]:
assert counter_method(Z) == expected
assert list_method(Z) == expected
assert gby_method(Z) == expected

In [27]:
%%timeit -r 30
counter_method(Z)

7.12 μs ± 40.2 ns per loop (mean ± std. dev. of 30 runs, 100,000 loops each)


In [28]:
%%timeit -r 30
list_method(Z)

12.3 ms ± 206 μs per loop (mean ± std. dev. of 30 runs, 100 loops each)


In [29]:
%%timeit -r 30
gby_method(Z)

5.18 μs ± 38 ns per loop (mean ± std. dev. of 30 runs, 100,000 loops each)


## Seven-Digit Highly-Composite
See https://oeis.org/A002182

In [30]:
Z = 1_441_440
expected = ((2, 5), (3, 2), (5, 1), (7, 1), (11, 1),(13, 1))

In [31]:
assert counter_method(Z) == expected
assert list_method(Z) == expected
assert gby_method(Z) == expected

In [32]:
%%timeit -r 30
counter_method(Z)

12.4 μs ± 121 ns per loop (mean ± std. dev. of 30 runs, 100,000 loops each)


In [33]:
%%timeit -r 30
list_method(Z)

169 ms ± 3.89 ms per loop (mean ± std. dev. of 30 runs, 10 loops each)


In [34]:
%%timeit -r 30
gby_method(Z)

10.2 μs ± 79.2 ns per loop (mean ± std. dev. of 30 runs, 100,000 loops each)


## Largest Prime under 1 Million

In [35]:
Z = 999_983
expected = ((Z,1),)

In [36]:
assert counter_method(Z) == expected
assert list_method(Z) == expected
assert gby_method(Z) == expected

In [37]:
%%timeit -r 30
counter_method(Z)

33.9 μs ± 357 ns per loop (mean ± std. dev. of 30 runs, 10,000 loops each)


In [38]:
%%timeit -r 30
list_method(Z)

81.5 ms ± 1.26 ms per loop (mean ± std. dev. of 30 runs, 10 loops each)


In [39]:
%%timeit -r 30
gby_method(Z)

33.8 μs ± 542 ns per loop (mean ± std. dev. of 30 runs, 10,000 loops each)
